# Model Training

This notebook shows the generalized sklearn-style model training workflow. It assumes the weekly model feature table has already been built by the data pipeline or `04_feature-engineering.ipynb`.

In [ ]:
from pathlib import Path

import pandas as pd

from gas_forecast.modeling import (
    DEFAULT_FEATURE_COLUMNS,
    DEFAULT_TARGET_COLUMN,
    ExpandingWindowSplitter,
    HoldoutSplitter,
    run_backtest,
    sklearn_model_configs,
)
from gas_forecast.data.paths import latest_processed_path
from gas_forecast.plotting import plot_weekly_change_forecast


In [ ]:
REGION = "R48"
PROCESSED_DIR = Path("../datasets/processed")
FEATURES_PATH = latest_processed_path(REGION, "weekly_model_features", PROCESSED_DIR)

features = pd.read_parquet(FEATURES_PATH)
feature_cols = list(DEFAULT_FEATURE_COLUMNS)
target_col = DEFAULT_TARGET_COLUMN
sklearn_configs = {config.key: config for config in sklearn_model_configs()}
features[["date", target_col, *feature_cols]].tail()


## Holdout Backtest

The holdout splitter gives one train/validation split. This matches the older notebook habit of comparing model output against a single recent year or period.

In [ ]:
holdout = HoldoutSplitter(
    date_col="date",
    train_end="2024-12-31",
    val_start="2025-01-01",
)
linear_config = sklearn_configs["linear_regression"]

linear_predictions, linear_metrics = run_backtest(
    features,
    feature_cols=feature_cols,
    target_col=target_col,
    date_col="date",
    model=linear_config.build(),
    splitter=holdout,
)

linear_metrics


In [ ]:
plot_weekly_change_forecast(
    linear_predictions,
    model_name=f"{linear_config.label} Holdout",
).show()


## Expanding-Window Backtest

The same `run_backtest` function works with a different validation strategy and a different sklearn estimator.

In [ ]:
expanding = ExpandingWindowSplitter(
    date_col="date",
    initial_train_start="2010-01-01",
    initial_train_end="2020-12-31",
    val_weeks=52,
    step_weeks=52,
)

configured_model_keys = [
    "ridge",
    "elastic_net",
    "random_forest",
    "hist_gradient_boosting",
]

model_results = {}
for key in configured_model_keys:
    config = sklearn_configs[key]
    predictions, metrics = run_backtest(
        features,
        feature_cols=feature_cols,
        target_col=target_col,
        date_col="date",
        model=config.build(),
        splitter=expanding,
    )
    model_results[key] = {
        "config": config,
        "predictions": predictions,
        "metrics": metrics,
    }

model_metrics = pd.concat(
    [
        result["metrics"].assign(model=result["config"].label)
        for result in model_results.values()
    ],
    ignore_index=True,
)
model_metrics


In [ ]:
tree_result = model_results["hist_gradient_boosting"]
plot_weekly_change_forecast(
    tree_result["predictions"],
    model_name=f"{tree_result['config'].label} Expanding CV",
).show()
